In [1]:
!pip install mlflow
!pip install gtts
!pip uninstall -y click
!pip install click==8.1.7
!pip install fastapi==0.110.0 uvicorn==0.27.1 typer==0.9.0 click==8.1.7
!pip uninstall -y click typer
!pip install click==8.1.7 typer==0.9.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found exis

In [2]:
"""
=============================================================================
 Multimodal Emotion AI — Production-Grade Pipeline v7.0
 Target: Autism Spectrum Disorder (ASD) Support System
 Modalities: Facial (EfficientNetB0) + Audio (MFCC-CNN)
 Explainability: Full Grad-CAM (vision, 4 artefacts) + SHAP (audio)
 Deployment: Gradio UI

 REWRITE NOTES v6.0 → v7.0
 ─────────────────────────────────────────────────────────────────────
 FIX-01  ModelCheckpoint: removed invalid `save_format` kwarg entirely.
         Keras controls format via file extension only:
           • ".keras" → Keras v3 native format  (TF ≥ 2.12, recommended)
           • ".h5"    → legacy HDF5             (always available)
         No save_format= argument is ever passed to ModelCheckpoint.

 FIX-02  Final model saves use model.save(path) only — not the older
         tf.saved_model.save() API which triggers _DictWrapper errors
         on multi-output / custom-layer models.

 FIX-03  Checkpoint directories are created with parents=True,
         exist_ok=True before any ModelCheckpoint is constructed.

 FIX-04  All training functions return the trained tf.keras.Model
         (not history, not a wrapper). History objects are discarded
         unless explicitly needed for plotting.

 FIX-05  _assert_keras_model() guards every save/GradCAM entry point.
         Raises RuntimeError with a clear message on type mismatch.

 FIX-06  Lambda layers fully replaced by named Layer subclasses with
         get_config() — required for model.save() with custom graphs.

 FIX-07  GradientTape scope: class_loss is selected from the raw logit
         tensor INSIDE the tape context, BEFORE any .numpy() call,
         ensuring gradients are never None.

 FIX-08  Joint generator: iter()-based face loop, correct float32 label
         blending, explicit shape assertions.

 FIX-09  CosineDecay training stage: ReduceLROnPlateau removed from
         callbacks when a schedule-based LR is already in use.

 FIX-10  Grad-CAM and SHAP outputs are written to disk only.
         Gradio UI is unchanged — shows text path summary.

 FIX-11  SHAP output normalisation handles all known return shapes from
         DeepExplainer across different TF/SHAP version combinations.

 FIX-12  collect_val_arrays uses batch-count guard (generators cycle
         infinitely; iterating on .n // batch_size is unreliable).
=============================================================================
"""

# ── stdlib ────────────────────────────────────────────────────────────────────
import os
import sys
import json
import zipfile
import datetime
import warnings
import logging
import traceback
from pathlib import Path
from typing import Optional, Dict, List, Tuple, Any

# ── numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── plotting ──────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ── deep learning ─────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, applications, callbacks, regularizers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    matthews_corrcoef, confusion_matrix, classification_report,
)
from sklearn.preprocessing import label_binarize

# ── audio / vision / explainability ──────────────────────────────────────────
import librosa
import shap
import cv2
from gtts import gTTS
import gradio as gr

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


# =============================================================================
# 0.  LOGGING
# =============================================================================

def setup_logging(log_dir: Path, debug: bool = False) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / "pipeline.log"
    level = logging.DEBUG if debug else logging.INFO
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)-25s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    root_logger = logging.getLogger("EmotionAI")
    root_logger.setLevel(level)
    if not root_logger.handlers:
        ch = logging.StreamHandler(sys.stdout)
        ch.setLevel(level)
        ch.setFormatter(fmt)
        root_logger.addHandler(ch)
        fh = logging.FileHandler(str(log_path), encoding="utf-8")
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(fmt)
        root_logger.addHandler(fh)
    root_logger.info(f"Logging initialised → {log_path}")
    return root_logger


logger = logging.getLogger("EmotionAI")


# =============================================================================
# 1.  CONSTANTS / MAPPINGS
# =============================================================================

PROJECT_NAME = "EmotionallyAdaptiveAI"

RAVDESS_TO_ASD: Dict[str, str] = {
    "01": "Natural",
    "03": "joy",
    "04": "sadness",
    "05": "anger",
    "06": "fear",
    "08": "surprise",
}

TARGET_EMOTIONS: List[str] = ["anger", "fear", "joy", "Natural", "sadness", "surprise"]
EMOTION_TO_IDX: Dict[str, int] = {e: i for i, e in enumerate(TARGET_EMOTIONS)}
IDX_TO_EMOTION: Dict[int, str] = {i: e for e, i in EMOTION_TO_IDX.items()}
NUM_CLASSES: int = len(TARGET_EMOTIONS)

N_MFCC: int = 40
AUDIO_SR: int = 16_000
AUDIO_DUR: float = 3.0
AUDIO_SHAPE: Tuple[int, int, int] = (N_MFCC, 128, 3)
IMG_SIZE: int = 224


# =============================================================================
# 2.  MODEL VALIDATION HELPER
# =============================================================================

def _assert_keras_model(obj: Any, label: str = "model") -> None:
    """Raise RuntimeError with a clear message if obj is not a tf.keras.Model."""
    if not isinstance(obj, tf.keras.Model):
        raise RuntimeError(
            f"Expected tf.keras.Model for '{label}', "
            f"got {type(obj).__name__!r}. "
            "Pass the model object directly — not a history, dict, or wrapper."
        )


# =============================================================================
# 3.  CUSTOM SERIALISABLE LAYERS  (FIX-06)
#     Every Lambda in the joint model is replaced by a named subclass
#     that implements get_config() so model.save() never needs closures.
# =============================================================================

class WeightedMul(layers.Layer):
    """Element-wise  x * w."""
    def call(self, inputs):
        x, w = inputs
        return x * w

    def get_config(self):
        return super().get_config()


class WeightedMulComplement(layers.Layer):
    """Element-wise  x * (1 - w)."""
    def call(self, inputs):
        x, w = inputs
        return x * (1.0 - w)

    def get_config(self):
        return super().get_config()


class L1Normalise(layers.Layer):
    """Divide each row by its L1 norm."""
    def call(self, x):
        return x / (tf.reduce_sum(x, axis=-1, keepdims=True) + 1e-9)

    def get_config(self):
        return super().get_config()


# =============================================================================
# 4.  ARTIFACT MANAGER
# =============================================================================

class ArtifactManager:
    """Centralised artefact export with retry, validation, and audit logging."""

    MAX_RETRIES = 3

    def __init__(self, base_dir: Path):
        self.base_dir = base_dir
        self._log = logging.getLogger("EmotionAI.ArtifactManager")
        for subdir in (
            "reports/explainability/gradcam",
            "reports/explainability/shap",
            "reports/metrics",
        ):
            (base_dir / subdir).mkdir(parents=True, exist_ok=True)
        self._log.info("Artefact directories ensured.")

    def _ts(self) -> str:
        return datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")

    def _validate(self, path: Path) -> bool:
        if not path.exists():
            self._log.error(f"Validation FAILED — not found: {path}")
            return False
        if path.stat().st_size == 0:
            self._log.error(f"Validation FAILED — empty file: {path}")
            return False
        self._log.debug(f"Validated {path.name} ({path.stat().st_size} B)")
        return True

    def _retry(self, fn, path: Path) -> Path:
        for attempt in range(1, self.MAX_RETRIES + 1):
            try:
                fn()
                if self._validate(path):
                    return path
                raise IOError("Post-save validation failed")
            except Exception as exc:
                self._log.warning(f"Attempt {attempt}/{self.MAX_RETRIES}: {exc}")
                if attempt == self.MAX_RETRIES:
                    raise
        raise RuntimeError(f"Could not save: {path}")

    def save_figure(self, fig, name: str, subdir: str,
                    modality: str = "joint", dpi: int = 120, ext: str = "png") -> Path:
        out_dir = self.base_dir / subdir
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{name}_{modality}_{self._ts()}.{ext}"

        def _w():
            fig.savefig(str(path), dpi=dpi, bbox_inches="tight")
            plt.close(fig)

        result = self._retry(_w, path)
        self._log.info(f"[SAVED] figure → {result}")
        return result

    def save_image_array(self, img: np.ndarray, name: str,
                         subdir: str, modality: str = "vision") -> Path:
        if img.dtype != np.uint8:
            img = np.clip(img, 0, 255).astype(np.uint8)
        out_dir = self.base_dir / subdir
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{name}_{modality}_{self._ts()}.png"

        def _w():
            ok = cv2.imwrite(str(path), img)
            if not ok:
                raise IOError("cv2.imwrite returned False")

        result = self._retry(_w, path)
        self._log.info(f"[SAVED] image → {result}")
        return result

    def save_json(self, data, name: str, subdir: str) -> Path:
        out_dir = self.base_dir / subdir
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{name}_{self._ts()}.json"

        def _w():
            with open(path, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4)

        result = self._retry(_w, path)
        self._log.info(f"[SAVED] json → {result}")
        return result

    def save_text(self, text: str, name: str, subdir: str) -> Path:
        out_dir = self.base_dir / subdir
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{name}_{self._ts()}.txt"

        def _w():
            with open(path, "w", encoding="utf-8") as f:
                f.write(text)

        result = self._retry(_w, path)
        self._log.info(f"[SAVED] text → {result}")
        return result

    def save_npz(self, arrays: dict, name: str, subdir: str) -> Path:
        out_dir = self.base_dir / subdir
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{name}_{self._ts()}.npz"

        def _w():
            np.savez_compressed(str(path), **arrays)

        result = self._retry(_w, path)
        self._log.info(f"[SAVED] npz → {result}")
        return result

    def save_csv(self, df: pd.DataFrame, name: str, subdir: str) -> Path:
        out_dir = self.base_dir / subdir
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{name}_{self._ts()}.csv"

        def _w():
            df.to_csv(str(path), index=False)

        result = self._retry(_w, path)
        self._log.info(f"[SAVED] csv → {result}")
        return result


# =============================================================================
# 5.  PROJECT STRUCTURE
# =============================================================================

def generate_project_structure(base: Path) -> Path:
    subdirs = [
        "src/models", "src/preprocessing", "src/utils", "src/engine",
        "data/raw/facial", "data/raw/audio", "data/processed", "data/memory",
        "reports/visualizations", "reports/explainability/gradcam",
        "reports/explainability/shap", "reports/metrics",
        "models/checkpoints/vision/stage1",
        "models/checkpoints/vision/stage2",
        "models/checkpoints/audio",
        "models/checkpoints/joint",
        "models/final",
        "outputs/audio", "outputs/schedules",
        "deployment/api", "deployment/docker", "logs",
    ]
    for s in subdirs:
        (base / s).mkdir(parents=True, exist_ok=True)

    files = {
        "README.md": (
            "# Multimodal Emotion AI v7.0\n"
            "Adaptive emotion recognition for ASD.\n\n"
            "## Run\n```bash\npython multimodal_emotion_ai.py\n```"
        ),
        "requirements.txt": (
            "tensorflow>=2.12\ngradio>=4.0\nlibrosa>=0.10\n"
            "scikit-learn>=1.2\nshap>=0.42\nopencv-python-headless\n"
            "gTTS\nsoundfile\naudioread\nmatplotlib\nseaborn\nnumpy\npandas"
        ),
    }
    for fname, content in files.items():
        fpath = base / fname
        if not fpath.exists():
            fpath.write_text(content)

    logger.info(f"Repository structure ready at: {base}")
    return base


# =============================================================================
# 6.  DATA LOADING
# =============================================================================

def parse_ravdess_label(filepath: str) -> Optional[str]:
    parts = Path(filepath).stem.split("-")
    return RAVDESS_TO_ASD.get(parts[2]) if len(parts) >= 3 else None


def extract_mfcc_spectrogram(filepath: str) -> Optional[np.ndarray]:
    _log = logging.getLogger("EmotionAI.audio_preprocess")
    try:
        y, _ = librosa.load(filepath, sr=AUDIO_SR, duration=AUDIO_DUR, mono=True)
        tlen = int(AUDIO_SR * AUDIO_DUR)
        y = np.pad(y, (0, max(0, tlen - len(y))))[:tlen]

        mfcc   = cv2.resize(
            librosa.feature.mfcc(y=y, sr=AUDIO_SR, n_mfcc=N_MFCC), (128, N_MFCC)
        )
        delta1 = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)

        def _norm(x):
            s = x.std()
            return (x - x.mean()) / (s if s >= 1e-9 else 1.0)

        arr = np.stack(
            [_norm(mfcc), _norm(delta1), _norm(delta2)], axis=-1
        ).astype(np.float32)

        if arr.shape != AUDIO_SHAPE or np.isnan(arr).any() or np.isinf(arr).any():
            return None
        return arr
    except Exception as exc:
        _log.error(f"MFCC error ({Path(filepath).name}): {exc}")
        return None


def load_audio_dataset(
    zip_path: str, base_dir: Path
) -> Tuple[np.ndarray, np.ndarray]:
    _log = logging.getLogger("EmotionAI.data_loader")
    extract_path = base_dir / "data/raw/audio"
    extract_path.mkdir(parents=True, exist_ok=True)

    if not Path(zip_path).exists():
        raise FileNotFoundError(f"Audio zip not found: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

    wav_files = list(extract_path.rglob("*.wav"))
    _log.info(f"Found {len(wav_files)} .wav files")

    X, y, skipped = [], [], 0
    for fp in wav_files:
        label = parse_ravdess_label(str(fp))
        if label is None or label not in EMOTION_TO_IDX:
            skipped += 1
            continue
        feat = extract_mfcc_spectrogram(str(fp))
        if feat is None:
            skipped += 1
            continue
        X.append(feat)
        y.append(EMOTION_TO_IDX[label])

    if not X:
        raise ValueError("No valid audio samples loaded.")

    _log.info(f"Audio: {len(X)} samples loaded, {skipped} skipped")
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


def load_facial_dataset(zip_path: str, base_dir: Path):
    _log = logging.getLogger("EmotionAI.data_loader")
    extract_path = base_dir / "data/raw/facial"
    extract_path.mkdir(parents=True, exist_ok=True)

    if not Path(zip_path).exists():
        raise FileNotFoundError(f"Facial zip not found: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

    train_dir = extract_path / "Train"
    test_dir  = extract_path / "Test"

    if not train_dir.exists():
        subs = [d for d in extract_path.rglob("Train") if d.is_dir()]
        if not subs:
            raise FileNotFoundError(f"No 'Train' folder found under {extract_path}")
        train_dir = subs[0]
        test_dir  = subs[0].parent / "Test"

    aug = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=25,
        width_shift_range=0.25,
        height_shift_range=0.25,
        shear_range=0.20,
        zoom_range=0.25,
        horizontal_flip=True,
        brightness_range=(0.7, 1.3),
        channel_shift_range=20.0,
        fill_mode="nearest",
    )
    val_gen_base = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )

    train_gen = aug.flow_from_directory(
        train_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=32,
        class_mode="categorical",
        classes=TARGET_EMOTIONS,
        shuffle=True,
        seed=SEED,
    )
    val_gen = val_gen_base.flow_from_directory(
        test_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=32,
        class_mode="categorical",
        classes=TARGET_EMOTIONS,
        shuffle=False,
    )

    cls_w = compute_class_weight(
        "balanced",
        classes=np.unique(train_gen.classes),
        y=train_gen.classes,
    )
    _log.info(f"Facial: train={train_gen.n}, val={val_gen.n}")
    return train_gen, val_gen, dict(enumerate(cls_w))


# =============================================================================
# 7.  MODEL DEFINITIONS
# =============================================================================

def build_vision_model(num_classes: int = NUM_CLASSES) -> tf.keras.Model:
    """EfficientNetB0 backbone with dual-dense head. Outputs raw logits."""
    base = applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        name="efficientnetb0",
    )
    base.trainable = False

    inp  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="facial_input")
    x    = base(inp, training=False)
    x    = layers.GlobalAveragePooling2D()(x)
    x    = layers.BatchNormalization()(x)
    x    = layers.Dropout(0.4)(x)
    x    = layers.Dense(512, activation="relu",
                        kernel_regularizer=regularizers.l2(1e-4))(x)
    x    = layers.BatchNormalization()(x)
    x    = layers.Dropout(0.3)(x)
    skip = layers.Dense(256, activation="relu",
                        kernel_regularizer=regularizers.l2(1e-4))(x)
    skip = layers.BatchNormalization()(skip)
    xp   = layers.Dense(256, use_bias=False)(x)
    x    = layers.Add()([skip, xp])
    x    = layers.Activation("relu")(x)
    x    = layers.Dropout(0.2)(x)
    out  = layers.Dense(num_classes, name="face_logits")(x)

    model = tf.keras.Model(inp, out, name="VisionModel")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(
            from_logits=True, label_smoothing=0.1
        ),
        metrics=["accuracy"],
    )
    logger.info("VisionModel built — EfficientNetB0, raw logits.")
    return model


def build_audio_model(num_classes: int = NUM_CLASSES) -> tf.keras.Model:
    """3-channel MFCC CNN. Outputs raw logits."""
    inp = tf.keras.Input(shape=AUDIO_SHAPE, name="audio_input")
    x   = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling2D((2, 2))(x)
    x   = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling2D((2, 2))(x)
    x   = layers.Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.SpatialDropout2D(0.3)(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation="relu",
                       kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, name="audio_logits")(x)

    model = tf.keras.Model(inp, out, name="AudioModel")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(
            from_logits=True, label_smoothing=0.1
        ),
        metrics=["accuracy"],
    )
    logger.info("AudioModel built — MFCC-CNN, raw logits.")
    return model


def build_softmax_wrapper(
    logit_model: tf.keras.Model, name: str
) -> tf.keras.Model:
    """Wrap a logit model with a Softmax output for probability inference."""
    _assert_keras_model(logit_model, "logit_model")
    inp = logit_model.input
    out = layers.Softmax(name="softmax_out")(logit_model.output)
    return tf.keras.Model(inp, out, name=name)


def build_joint_model(
    vision_model: tf.keras.Model,
    audio_model:  tf.keras.Model,
    num_classes:  int = NUM_CLASSES,
) -> Tuple[tf.keras.Model, tf.keras.Model]:
    """
    Attention-gated fusion of face and audio probabilities.
    Uses only named custom Layer subclasses (no Lambda) so model.save()
    works correctly.
    Returns (joint_train_model, joint_infer_model).
    """
    _assert_keras_model(vision_model, "vision_model")
    _assert_keras_model(audio_model, "audio_model")

    face_inp  = tf.keras.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3), name="joint_face_input"
    )
    audio_inp = tf.keras.Input(shape=AUDIO_SHAPE, name="joint_audio_input")

    face_logits  = vision_model(face_inp,  training=False)
    audio_logits = audio_model(audio_inp,  training=False)

    face_feat  = layers.Softmax(name="face_probs")(face_logits)
    audio_feat = layers.Softmax(name="audio_probs")(audio_logits)

    combined = layers.Concatenate(name="gate_concat")([face_feat, audio_feat])
    gate_h   = layers.Dense(64, activation="relu", name="gate_hidden")(combined)
    gate     = layers.Dense(1, activation="sigmoid", name="gate")(gate_h)

    # FIX-06: custom layers instead of Lambda
    face_w  = WeightedMul(name="face_weighted")([face_feat, gate])
    audio_w = WeightedMulComplement(name="audio_weighted")([audio_feat, gate])
    fused   = layers.Add(name="fused_probs")([face_w, audio_w])
    out     = L1Normalise(name="joint_output")(fused)

    joint_train = tf.keras.Model(
        inputs=[face_inp, audio_inp], outputs=out, name="JointModel_train"
    )
    joint_train.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    joint_infer = tf.keras.Model(
        inputs=[face_inp, audio_inp],
        outputs=[out, gate],
        name="JointModel_infer",
    )

    logger.info("JointModel built — custom serialisable layers, attention gate.")
    return joint_train, joint_infer


# =============================================================================
# 8.  MODEL SAVING HELPER  (FIX-02)
# =============================================================================

def save_model_keras(
    model: tf.keras.Model, save_path: Path, label: str
) -> Path:
    """
    FIX-02: use model.save(path) for final exports.
    Path extension controls format:
      .keras → Keras v3 native (recommended for TF ≥ 2.12)
      .h5    → legacy HDF5
    FIX-05: validates isinstance before saving.
    """
    _assert_keras_model(model, label)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        model.save(str(save_path))
        logger.info(f"[SAVED] {label} → {save_path}")
    except Exception as exc:
        raise RuntimeError(f"Failed to save {label} to {save_path}: {exc}") from exc
    return save_path


# =============================================================================
# 9.  TRAINING CALLBACKS  (FIX-01 / FIX-03)
# =============================================================================

def get_standard_callbacks(
    ckpt_dir: Path,
    monitor: str = "val_accuracy",
    use_reduce_lr: bool = True,
) -> List[callbacks.Callback]:
    """
    FIX-01: ModelCheckpoint uses .keras extension only — no save_format kwarg.
    FIX-03: checkpoint directory created before constructing callback.
    """
    # FIX-03: ensure directory exists first
    ckpt_dir = Path(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # FIX-01: extension controls format; no save_format= argument
    ckpt_path = str(ckpt_dir / "best_model.keras")

    cbs = [
        callbacks.ModelCheckpoint(
            filepath=ckpt_path,
            monitor=monitor,
            save_best_only=True,
            verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor=monitor,
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.TerminateOnNaN(),
    ]

    if use_reduce_lr:
        cbs.insert(
            2,
            callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.3,
                patience=3,
                min_lr=1e-7,
                verbose=1,
            ),
        )
    return cbs


# =============================================================================
# 10.  HYBRID FUSION
# =============================================================================

class HybridFusion:
    NOISE_FACE_W = 0.70
    DARK_FACE_W  = 0.30

    @staticmethod
    def assess_image_quality(img_bgr: np.ndarray) -> str:
        if img_bgr is None or img_bgr.size == 0:
            return "unknown"
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        if float(gray.mean()) < 40:
            return "dark"
        if float(cv2.Laplacian(gray, cv2.CV_64F).var()) < 50:
            return "blurry"
        return "normal"

    @staticmethod
    def fuse(
        face_probs: np.ndarray,
        audio_probs: np.ndarray,
        learned_gate: float,
        image_quality: str,
    ) -> Tuple[np.ndarray, float, float]:
        rule_w  = (
            HybridFusion.DARK_FACE_W
            if image_quality in ("dark", "blurry")
            else HybridFusion.NOISE_FACE_W
        )
        face_w  = 0.5 * rule_w + 0.5 * float(learned_gate)
        audio_w = 1.0 - face_w
        fused   = face_probs * face_w + audio_probs * audio_w
        total   = fused.sum()
        fused   = (
            fused / total if total >= 1e-9
            else np.ones_like(fused) / len(fused)
        )
        return fused, face_w, audio_w


# =============================================================================
# 11.  GRAD-CAM EXPLAINER
# =============================================================================

class GradCAMExplainer:
    """
    Guided Grad-CAM for EfficientNetB0 vision model.
    Saves FOUR artefacts per call; no Gradio visual output.

    Saved to reports/explainability/gradcam/
    ──────────────────────────────────────────────────────────
    (a) 3-panel PNG  — original | raw heatmap | overlay
    (b) Greyscale raw heatmap PNG
    (c) JET colourised heatmap PNG
    (d) Alpha-blended overlay PNG

    FIX-07: class_loss is selected from the logit tensor INSIDE the
    GradientTape context BEFORE any .numpy() call, so grads are
    never None.
    """

    PREFERRED_LAYER = "top_activation"

    def __init__(
        self,
        vision_model: tf.keras.Model,
        artifact_manager: ArtifactManager,
    ):
        self._log = logging.getLogger("EmotionAI.GradCAM")
        _assert_keras_model(vision_model, "vision_model for GradCAM")

        # Guard: reject softmax-wrapped models
        last_cfg = vision_model.layers[-1].get_config()
        if "softmax" in str(last_cfg.get("activation", "")).lower():
            raise RuntimeError(
                "GradCAMExplainer requires the raw logit model, "
                "not a softmax-wrapped version."
            )

        self.model     = vision_model
        self.artifacts = artifact_manager

        try:
            eff_base = self.model.get_layer("efficientnetb0")
        except ValueError as exc:
            raise RuntimeError(
                "Cannot find 'efficientnetb0' in vision model."
            ) from exc

        conv_layer = self._resolve_conv_layer(eff_base)
        self._log.info(f"Grad-CAM conv layer: {conv_layer.name}")

        # Gradient model outputs (conv_output, raw_logits)
        self.grad_model = tf.keras.Model(
            inputs=self.model.inputs,
            outputs=[conv_layer.output, self.model.output],
            name="GradCAM_grad_model",
        )

    def _resolve_conv_layer(self, base_model) -> layers.Layer:
        try:
            return base_model.get_layer(self.PREFERRED_LAYER)
        except ValueError:
            self._log.warning(
                f"'{self.PREFERRED_LAYER}' not found — falling back to last Conv2D"
            )
        conv_layers = [l for l in base_model.layers if isinstance(l, layers.Conv2D)]
        if not conv_layers:
            raise RuntimeError("No Conv2D layers found in EfficientNetB0.")
        return conv_layers[-1]

    def compute(
        self,
        img_array: np.ndarray,     # float32, shape (1, H, W, 3), preprocessed
        raw_rgb: np.ndarray,       # uint8,   shape (H, W, 3), original
        class_idx: Optional[int] = None,
    ) -> Dict[str, Any]:
        """
        Compute Guided Grad-CAM and save 4 artefacts.
        Returns result dict containing paths and arrays.
        """
        self._log.info("Computing Guided Grad-CAM …")

        if img_array.ndim != 4:
            raise ValueError(
                f"img_array must be 4-D (1,H,W,3), got shape {img_array.shape}"
            )
        if raw_rgb.ndim != 3 or raw_rgb.dtype != np.uint8:
            raise ValueError(
                f"raw_rgb must be uint8 (H,W,3), got {raw_rgb.dtype} {raw_rgb.shape}"
            )
        if raw_rgb.shape[:2] != (IMG_SIZE, IMG_SIZE):
            raise ValueError(
                f"raw_rgb must be ({IMG_SIZE},{IMG_SIZE},3), got {raw_rgb.shape}"
            )

        img_tf = tf.cast(img_array, tf.float32)

        # FIX-07: ALL tensor ops inside tape; NO .numpy() until after tape.gradient()
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(img_tf)
            conv_outputs, predictions = self.grad_model(img_tf, training=False)
            tape.watch(conv_outputs)

            if class_idx is None:
                class_idx_tf = tf.argmax(predictions[0], output_type=tf.int32)
            else:
                class_idx_tf = tf.constant(class_idx, dtype=tf.int32)

            class_loss = predictions[:, class_idx_tf]

        grads = tape.gradient(class_loss, conv_outputs)
        del tape

        class_idx = int(class_idx_tf.numpy())

        if grads is None:
            raise RuntimeError(
                "GradientTape returned None. "
                "Ensure the vision model outputs raw logits (no stop_gradient)."
            )

        grads_np  = grads[0].numpy()
        conv_np   = conv_outputs[0].numpy()
        probs_np  = tf.nn.softmax(predictions[0]).numpy()
        confidence = float(probs_np[class_idx])
        emotion    = IDX_TO_EMOTION[class_idx]

        grads_np = np.nan_to_num(grads_np, nan=0.0, posinf=1.0, neginf=-1.0)
        conv_np  = np.nan_to_num(conv_np,  nan=0.0, posinf=1.0, neginf=-1.0)

        gate_f  = (conv_np  > 0).astype(np.float32)
        gate_r  = (grads_np > 0).astype(np.float32)
        guided  = gate_f * gate_r * grads_np
        weights = guided.mean(axis=(0, 1))
        cam     = np.dot(conv_np, weights)
        cam     = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        cam     = np.maximum(cam, 0)

        cam_min, cam_max = cam.min(), cam.max()
        if (cam_max - cam_min) < 1e-10:
            self._log.warning("Near-zero Grad-CAM activation — returning blank heatmap.")
            cam = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
        else:
            cam = (cam - cam_min) / (cam_max - cam_min)

        raw_heatmap_u8    = np.uint8(255 * cam)
        heatmap_color_bgr = cv2.applyColorMap(raw_heatmap_u8, cv2.COLORMAP_JET)
        raw_bgr           = cv2.cvtColor(raw_rgb, cv2.COLOR_RGB2BGR)
        overlay_bgr = np.clip(
            cv2.addWeighted(raw_bgr, 0.6, heatmap_color_bgr, 0.4, 0), 0, 255
        ).astype(np.uint8)
        overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

        subdir = "reports/explainability/gradcam"

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(
            f"Grad-CAM  |  Predicted: {emotion.upper()}  |  "
            f"Confidence: {confidence:.2%}",
            fontsize=13, fontweight="bold",
        )
        axes[0].imshow(raw_rgb)
        axes[0].set_title("Original Image")
        axes[0].axis("off")
        im = axes[1].imshow(cam, cmap="jet", vmin=0, vmax=1)
        axes[1].set_title("Grad-CAM Heatmap")
        axes[1].axis("off")
        plt.colorbar(
            plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(0, 1)),
            ax=axes[1], fraction=0.046, pad=0.04,
        ).set_label("Activation")
        axes[2].imshow(overlay_rgb)
        axes[2].set_title("Guided Overlay")
        axes[2].axis("off")
        plt.tight_layout()

        report_path = self.artifacts.save_figure(
            fig, f"gradcam_report_{emotion}", subdir, modality="vision"
        )
        raw_heatmap_path = self.artifacts.save_image_array(
            cv2.cvtColor(raw_heatmap_u8, cv2.COLOR_GRAY2BGR),
            f"gradcam_raw_heatmap_{emotion}", subdir, modality="vision",
        )
        colour_heatmap_path = self.artifacts.save_image_array(
            heatmap_color_bgr,
            f"gradcam_colour_heatmap_{emotion}", subdir, modality="vision",
        )
        overlay_path = self.artifacts.save_image_array(
            overlay_bgr,
            f"gradcam_overlay_{emotion}", subdir, modality="vision",
        )

        self._log.info(
            f"[GRADCAM] {emotion} ({confidence:.2%})\n"
            f"  report         → {report_path}\n"
            f"  raw heatmap    → {raw_heatmap_path}\n"
            f"  colour heatmap → {colour_heatmap_path}\n"
            f"  overlay        → {overlay_path}"
        )

        result = {
            "heatmap":             cam,
            "overlay_rgb":         overlay_rgb,
            "class_idx":           class_idx,
            "emotion":             emotion,
            "confidence":          confidence,
            "report_path":         str(report_path),
            "raw_heatmap_path":    str(raw_heatmap_path),
            "colour_heatmap_path": str(colour_heatmap_path),
            "overlay_path":        str(overlay_path),
        }
        self._validate_result(result)
        return result

    def _validate_result(self, d: dict) -> None:
        required = [
            "heatmap", "overlay_rgb", "emotion", "confidence",
            "report_path", "raw_heatmap_path", "colour_heatmap_path", "overlay_path",
        ]
        for key in required:
            if key not in d or d[key] is None:
                raise RuntimeError(f"GradCAM result missing field: '{key}'")
        ov = d["overlay_rgb"]
        if not (
            isinstance(ov, np.ndarray)
            and ov.dtype == np.uint8
            and ov.ndim == 3
            and ov.shape[2] == 3
        ):
            raise RuntimeError(
                f"overlay_rgb must be uint8 (H,W,3), "
                f"got {getattr(ov,'shape',None)} {getattr(ov,'dtype',None)}"
            )


# =============================================================================
# 12.  SHAP AUDIO EXPLAINER  (FIX-11)
# =============================================================================

class SHAPAudioExplainer:
    """
    SHAP DeepExplainer for the audio CNN.
    FIX-11: normalise_shap_output() handles all known return shapes
            across TF/SHAP version combinations.
    Saves plots to disk; no Gradio output.
    """

    def __init__(
        self,
        audio_logit_model: tf.keras.Model,
        background_data:   np.ndarray,
        artifact_manager:  ArtifactManager,
        n_background:      int = 50,
    ):
        self._log = logging.getLogger("EmotionAI.SHAP")
        _assert_keras_model(audio_logit_model, "audio_logit_model")

        self.logit_model = audio_logit_model
        self.prob_model  = build_softmax_wrapper(
            audio_logit_model, name="AudioModel_shap_prob"
        )
        self.artifacts   = artifact_manager

        if background_data.ndim == 3:
            background_data = background_data[np.newaxis]
        if background_data.ndim != 4 or len(background_data) == 0:
            raise ValueError(
                f"background_data must be non-empty 4-D, got {background_data.shape}"
            )

        n   = min(n_background, len(background_data))
        idx = np.random.choice(len(background_data), n, replace=False)
        self.background = background_data[idx].astype(np.float32)
        self._log.info(
            f"SHAP background: {n} samples {self.background.shape}"
        )

        try:
            self.explainer = shap.DeepExplainer(
                self.logit_model, self.background
            )
            self._log.info("SHAP DeepExplainer ready.")
        except Exception as exc:
            raise RuntimeError("Could not initialise SHAP explainer.") from exc

    def _normalize_shap_output(
        self, raw: Any, n_samples: int
    ) -> List[np.ndarray]:
        if isinstance(raw, list) and len(raw) == 0:
            self._log.warning("SHAP returned empty list — substituting zeros.")
            return [
                np.zeros((n_samples, *AUDIO_SHAPE), dtype=np.float32)
                for _ in range(NUM_CLASSES)
            ]

        if isinstance(raw, list) and len(raw) == NUM_CLASSES:
            result = [np.array(sv, dtype=np.float32) for sv in raw]

        elif isinstance(raw, list) and len(raw) == 1:
            arr = np.array(raw[0], dtype=np.float32)
            if arr.ndim == 5 and arr.shape[0] == NUM_CLASSES:
                result = [arr[c] for c in range(NUM_CLASSES)]
            else:
                result = [arr.copy() for _ in range(NUM_CLASSES)]

        elif isinstance(raw, np.ndarray):
            if raw.ndim == 5 and raw.shape[0] == NUM_CLASSES:
                result = [raw[c].astype(np.float32) for c in range(NUM_CLASSES)]
            elif raw.ndim == 5 and raw.shape[-1] == NUM_CLASSES:
                result = [raw[..., c].astype(np.float32) for c in range(NUM_CLASSES)]
            elif raw.ndim == 4:
                result = [raw.astype(np.float32).copy() for _ in range(NUM_CLASSES)]
            else:
                raise RuntimeError(
                    f"Unrecognised SHAP ndarray shape: {raw.shape}"
                )
        else:
            raise RuntimeError(
                f"Unrecognised SHAP output type: {type(raw)}"
            )

        for i, sv in enumerate(result):
            if sv.shape[0] != n_samples:
                raise RuntimeError(
                    f"shap_values[{i}] sample count "
                    f"{sv.shape[0]} ≠ {n_samples}"
                )
            result[i] = np.nan_to_num(sv, nan=0.0, posinf=0.0, neginf=0.0)

        return result

    def explain(
        self,
        sample: np.ndarray,
        true_label: Optional[int] = None,
    ) -> Dict[str, Any]:
        self._log.info("Computing SHAP for audio sample …")

        if sample.ndim == 3:
            sample = sample[np.newaxis]
        if sample.shape[1:] != AUDIO_SHAPE:
            raise ValueError(
                f"Sample shape {sample.shape[1:]} ≠ expected {AUDIO_SHAPE}"
            )
        sample = sample.astype(np.float32)

        try:
            raw_shap = self.explainer.shap_values(sample)
        except Exception as exc:
            raise RuntimeError("SHAP explanation failed.") from exc

        shap_values = self._normalize_shap_output(raw_shap, n_samples=1)
        pred_probs  = self.prob_model.predict(sample, verbose=0)[0]
        pred_class  = int(np.argmax(pred_probs))
        emotion     = IDX_TO_EMOTION[pred_class]
        confidence  = float(pred_probs[pred_class])
        sv_pred     = shap_values[pred_class][0, :, :, 0]
        subdir      = "reports/explainability/shap"

        fig1, ax1 = plt.subplots(1, 2, figsize=(14, 5))
        fig1.suptitle(
            f"SHAP Audio  |  {emotion.upper()}  |  {confidence:.2%}", fontsize=12
        )
        ax1[0].imshow(
            sample[0, :, :, 0], aspect="auto", origin="lower", cmap="magma"
        )
        ax1[0].set_title("MFCC Input (ch0)")
        ax1[0].set_xlabel("Time"); ax1[0].set_ylabel("MFCC")
        vm = max(np.abs(sv_pred).max(), 1e-10)
        im = ax1[1].imshow(
            sv_pred, aspect="auto", origin="lower",
            cmap="RdBu", vmin=-vm, vmax=vm,
        )
        ax1[1].set_title(f"SHAP — {emotion}")
        ax1[1].set_xlabel("Time"); ax1[1].set_ylabel("MFCC")
        plt.colorbar(im, ax=ax1[1])
        plt.tight_layout()
        plot_path = self.artifacts.save_figure(
            fig1, f"shap_{emotion}", subdir, modality="audio"
        )

        fig2, ax2 = plt.subplots(figsize=(8, 4))
        ax2.barh(range(N_MFCC), np.abs(sv_pred).mean(axis=1), color="steelblue")
        ax2.set_xlabel("Mean |SHAP|")
        ax2.set_ylabel("MFCC Coeff")
        ax2.set_title(f"SHAP Feature Importance — {emotion.upper()}")
        plt.tight_layout()
        bar_path = self.artifacts.save_figure(
            fig2, f"shap_bar_{emotion}", subdir, modality="audio"
        )

        fig3, ax3 = plt.subplots(2, 3, figsize=(18, 10))
        for cidx, emo in IDX_TO_EMOTION.items():
            r, c = divmod(cidx, 3)
            sv   = shap_values[cidx][0, :, :, 0]
            vm_c = max(np.abs(sv).max(), 1e-10)
            ax3[r, c].imshow(
                sv, aspect="auto", origin="lower",
                cmap="RdBu", vmin=-vm_c, vmax=vm_c,
            )
            ax3[r, c].set_title(emo.upper())
            ax3[r, c].axis("off")
        fig3.suptitle("SHAP — All Classes", fontsize=14)
        plt.tight_layout()
        allclass_path = self.artifacts.save_figure(
            fig3, f"shap_allclass_{emotion}", subdir, modality="audio"
        )

        npz_path = self.artifacts.save_npz(
            {IDX_TO_EMOTION[i]: shap_values[i][0, :, :, 0]
             for i in range(NUM_CLASSES)},
            f"shap_values_{emotion}", subdir,
        )

        self._log.info(
            f"[SHAP] {emotion} ({confidence:.2%}) | "
            f"plot={plot_path} | bar={bar_path}"
        )
        return {
            "shap_values":    shap_values,
            "pred_class":     pred_class,
            "emotion":        emotion,
            "confidence":     confidence,
            "plot_path":      str(plot_path),
            "bar_path":       str(bar_path),
            "allclass_path":  str(allclass_path),
            "npz_path":       str(npz_path),
        }

    def batch_shap_report(
        self, X: np.ndarray, y: np.ndarray, n_samples: int = 20
    ) -> Tuple[Path, Path]:
        n     = min(n_samples, len(X))
        X_sub = X[np.random.choice(len(X), n, replace=False)].astype(np.float32)

        try:
            raw = self.explainer.shap_values(X_sub)
        except Exception as exc:
            raise RuntimeError("Batch SHAP failed.") from exc

        sv_all = self._normalize_shap_output(raw, n_samples=n)
        rows   = []
        for c in range(NUM_CLASSES):
            sv_c = sv_all[c][:, :, :, 0]
            for i, v in enumerate(np.abs(sv_c).mean(axis=(0, 2))):
                rows.append({
                    "emotion":       IDX_TO_EMOTION[c],
                    "mfcc_coeff":    i,
                    "mean_abs_shap": float(v),
                })

        df    = pd.DataFrame(rows)
        subdir = "reports/explainability/shap"
        csv   = self.artifacts.save_csv(df, "shap_importance_summary", subdir)

        pivot = df.pivot(
            index="mfcc_coeff", columns="emotion", values="mean_abs_shap"
        )
        fig, ax = plt.subplots(figsize=(12, 8))
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax)
        ax.set_title("Mean |SHAP| per MFCC × Emotion")
        plt.tight_layout()
        heat = self.artifacts.save_figure(
            fig, "shap_importance_heatmap", subdir, modality="audio"
        )
        return csv, heat


# =============================================================================
# 13.  EVALUATION
# =============================================================================

def collect_val_arrays(
    val_gen, n_total: int
) -> Tuple[np.ndarray, np.ndarray]:
    """
    FIX-12: use n_batches = len(val_gen) guard; generators cycle
    infinitely so iterating on .n // batch_size is unreliable.
    """
    _log = logging.getLogger("EmotionAI.evaluation")
    val_gen.reset()
    X_list, y_list = [], []
    n_batches = len(val_gen)

    for batch_idx, (X_batch, y_batch) in enumerate(val_gen):
        X_list.append(X_batch)
        y_list.append(np.argmax(y_batch, axis=1))
        if batch_idx + 1 >= n_batches:
            break

    if not X_list:
        raise RuntimeError("val_gen yielded no batches.")

    X_all = np.concatenate(X_list, axis=0)[:n_total]
    y_all = np.concatenate(y_list, axis=0)[:n_total]
    _log.info(f"Collected val arrays: X={X_all.shape}, y={y_all.shape}")
    return X_all, y_all


def evaluate_and_report(
    model: tf.keras.Model,
    X: np.ndarray,
    y_true: np.ndarray,
    artifact_manager: ArtifactManager,
    model_name: str = "model",
) -> Dict[str, Any]:
    """
    Full evaluation: metrics, classification report, confusion matrix, ROC.
    Auto-detects logit vs softmax output.
    """
    _log = logging.getLogger("EmotionAI.evaluation")
    _assert_keras_model(model, model_name)
    _log.info(f"Evaluating {model_name} …")

    raw_preds = model.predict(X, verbose=0)

    if raw_preds.min() < -0.01 or raw_preds.max() > 1.01:
        _log.info("Logit output detected — applying softmax.")
        y_pred_probs = tf.nn.softmax(raw_preds).numpy()
    else:
        y_pred_probs = raw_preds

    y_pred = np.argmax(y_pred_probs, axis=1)

    acc   = accuracy_score(y_true, y_pred)
    prec  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec   = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1    = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mcc   = matthews_corrcoef(y_true, y_pred)

    y_bin   = label_binarize(y_true, classes=range(NUM_CLASSES))
    roc_auc = roc_auc_score(
        y_bin, y_pred_probs, average="weighted", multi_class="ovr"
    )

    pr_auc_list  = []
    for i in range(NUM_CLASSES):
        p, r, _ = precision_recall_curve(y_bin[:, i], y_pred_probs[:, i])
        pr_auc_list.append(auc(r, p))

    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)

    metrics = {
        "model":               model_name,
        "accuracy":            round(float(acc),  4),
        "precision_weighted":  round(float(prec), 4),
        "recall_weighted":     round(float(rec),  4),
        "f1_weighted":         round(float(f1),   4),
        "roc_auc_weighted":    round(float(roc_auc), 4),
        "pr_auc_mean":         round(float(np.mean(pr_auc_list)), 4),
        "mcc":                 round(float(mcc),  4),
        "f1_per_class": {
            TARGET_EMOTIONS[i]: round(float(f1_per_class[i]), 4)
            for i in range(NUM_CLASSES)
        },
        "pr_auc_per_class": {
            TARGET_EMOTIONS[i]: round(float(pr_auc_list[i]), 4)
            for i in range(NUM_CLASSES)
        },
    }

    for k, v in metrics.items():
        if k not in ("model", "f1_per_class", "pr_auc_per_class"):
            _log.info(f"  {k:<28}: {v:.4f}")

    report_txt = classification_report(
        y_true, y_pred,
        target_names=TARGET_EMOTIONS,
        zero_division=0,
    )
    _log.info(f"\n{report_txt}")

    artifact_manager.save_json(metrics, f"{model_name}_metrics", "reports/metrics")
    artifact_manager.save_text(report_txt, f"{model_name}_report", "reports/metrics")

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=TARGET_EMOTIONS,
        yticklabels=TARGET_EMOTIONS,
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix — {model_name}")
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")
    plt.tight_layout()
    artifact_manager.save_figure(
        fig, f"{model_name}_confusion_matrix", "reports/metrics"
    )

    fig2, ax2 = plt.subplots(figsize=(10, 7))
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_probs[:, i])
        ax2.plot(
            fpr, tpr,
            label=f"{TARGET_EMOTIONS[i]} (AUC={pr_auc_list[i]:.2f})",
        )
    ax2.plot([0, 1], [0, 1], "k--")
    ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR")
    ax2.set_title(f"ROC — {model_name}")
    ax2.legend()
    plt.tight_layout()
    artifact_manager.save_figure(fig2, f"{model_name}_roc", "reports/metrics")

    return metrics


# =============================================================================
# 13b.  JOINT MODEL EVALUATION
# =============================================================================

def evaluate_joint_model(
    joint_infer_model: tf.keras.Model,
    X_face_val:        np.ndarray,
    X_audio_val:       np.ndarray,
    y_true:            np.ndarray,
    artifact_manager:  ArtifactManager,
    model_name:        str = "JointModel",
) -> Dict[str, Any]:
    """
    Full evaluation of the joint inference model using paired face + audio inputs.
    Handles mismatched sample counts by aligning to the minimum length.
    Produces: metrics JSON, classification report, confusion matrix, ROC curves,
              PR curves, per-class F1 bar chart, and gate distribution plot.
    Auto-detects fused output shape from joint_infer_model (may return
    [fused_probs, gate] or just fused_probs).
    """
    _log = logging.getLogger("EmotionAI.evaluation")
    _assert_keras_model(joint_infer_model, model_name)
    _log.info(f"Evaluating {model_name} (joint face + audio) …")

    # ── Align sample counts ───────────────────────────────────────────────────
    n = min(len(X_face_val), len(X_audio_val), len(y_true))
    if n == 0:
        raise ValueError("No samples available for joint evaluation.")
    if n < max(len(X_face_val), len(X_audio_val), len(y_true)):
        _log.warning(
            f"Sample count mismatch — aligning to {n} "
            f"(face={len(X_face_val)}, audio={len(X_audio_val)}, "
            f"labels={len(y_true)})"
        )
    X_face_val  = X_face_val[:n]
    X_audio_val = X_audio_val[:n]
    y_true      = y_true[:n]

    # ── Predict ───────────────────────────────────────────────────────────────
    joint_inputs = {
        "joint_face_input":  X_face_val.astype(np.float32),
        "joint_audio_input": X_audio_val.astype(np.float32),
    }
    raw_out = joint_infer_model.predict(joint_inputs, verbose=0)

    # Joint infer model returns [fused_probs, gate] — extract fused_probs only
    if isinstance(raw_out, (list, tuple)):
        y_pred_probs = np.array(raw_out[0])
        gate_vals    = np.array(raw_out[1]).flatten() if len(raw_out) > 1 else None
        _log.info(
            f"Joint output unpacked — probs shape: {y_pred_probs.shape}"
            + (f", gate shape: {gate_vals.shape}" if gate_vals is not None else "")
        )
    else:
        y_pred_probs = np.array(raw_out)
        gate_vals    = None

    # Normalise to probabilities if logits slipped through
    if y_pred_probs.min() < -0.01 or y_pred_probs.max() > 1.01:
        _log.info("Logit output detected in joint model — applying softmax.")
        y_pred_probs = tf.nn.softmax(y_pred_probs).numpy()

    y_pred = np.argmax(y_pred_probs, axis=1)

    # ── Core metrics ──────────────────────────────────────────────────────────
    acc   = accuracy_score(y_true, y_pred)
    prec  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec   = recall_score(y_true, y_pred,    average="weighted", zero_division=0)
    f1    = f1_score(y_true, y_pred,        average="weighted", zero_division=0)
    mcc   = matthews_corrcoef(y_true, y_pred)

    y_bin   = label_binarize(y_true, classes=range(NUM_CLASSES))
    roc_auc = roc_auc_score(
        y_bin, y_pred_probs, average="weighted", multi_class="ovr"
    )

    pr_auc_list = []
    for i in range(NUM_CLASSES):
        p, r, _ = precision_recall_curve(y_bin[:, i], y_pred_probs[:, i])
        pr_auc_list.append(auc(r, p))

    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)

    # Gate statistics — unique insight of the joint model
    gate_stats = {}
    if gate_vals is not None:
        gate_stats = {
            "gate_mean": round(float(gate_vals.mean()), 4),
            "gate_std":  round(float(gate_vals.std()),  4),
            "gate_min":  round(float(gate_vals.min()),  4),
            "gate_max":  round(float(gate_vals.max()),  4),
        }
        _log.info(
            f"  gate_mean={gate_stats['gate_mean']:.4f}  "
            f"gate_std={gate_stats['gate_std']:.4f}"
        )

    metrics = {
        "model":               model_name,
        "n_samples":           int(n),
        "accuracy":            round(float(acc),     4),
        "precision_weighted":  round(float(prec),    4),
        "recall_weighted":     round(float(rec),     4),
        "f1_weighted":         round(float(f1),      4),
        "roc_auc_weighted":    round(float(roc_auc), 4),
        "pr_auc_mean":         round(float(np.mean(pr_auc_list)), 4),
        "mcc":                 round(float(mcc),     4),
        "f1_per_class": {
            TARGET_EMOTIONS[i]: round(float(f1_per_class[i]), 4)
            for i in range(NUM_CLASSES)
        },
        "pr_auc_per_class": {
            TARGET_EMOTIONS[i]: round(float(pr_auc_list[i]), 4)
            for i in range(NUM_CLASSES)
        },
        **gate_stats,
    }

    for k, v in metrics.items():
        if k not in (
            "model", "n_samples", "f1_per_class",
            "pr_auc_per_class",
        ):
            _log.info(f"  {k:<28}: {v}")

    report_txt = classification_report(
        y_true, y_pred,
        target_names=TARGET_EMOTIONS,
        zero_division=0,
    )
    _log.info(f"\n{report_txt}")

    subdir = "reports/metrics"
    artifact_manager.save_json(metrics,    f"{model_name}_metrics", subdir)
    artifact_manager.save_text(report_txt, f"{model_name}_report",  subdir)

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Purples",
        xticklabels=TARGET_EMOTIONS,
        yticklabels=TARGET_EMOTIONS,
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix — {model_name}")
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")
    plt.tight_layout()
    artifact_manager.save_figure(fig, f"{model_name}_confusion_matrix", subdir)

    # ── ROC curves ────────────────────────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(10, 7))
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_probs[:, i])
        ax2.plot(
            fpr, tpr,
            label=f"{TARGET_EMOTIONS[i]} (AUC={pr_auc_list[i]:.2f})",
        )
    ax2.plot([0, 1], [0, 1], "k--", label="Random")
    ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR")
    ax2.set_title(f"ROC Curves — {model_name}")
    ax2.legend(loc="lower right")
    plt.tight_layout()
    artifact_manager.save_figure(fig2, f"{model_name}_roc", subdir)

    # ── PR curves ─────────────────────────────────────────────────────────────
    fig3, ax3 = plt.subplots(figsize=(10, 7))
    for i in range(NUM_CLASSES):
        p, r, _ = precision_recall_curve(y_bin[:, i], y_pred_probs[:, i])
        ax3.plot(
            r, p,
            label=f"{TARGET_EMOTIONS[i]} (AUC={pr_auc_list[i]:.2f})",
        )
    ax3.set_xlabel("Recall"); ax3.set_ylabel("Precision")
    ax3.set_title(f"Precision-Recall Curves — {model_name}")
    ax3.legend(loc="upper right")
    plt.tight_layout()
    artifact_manager.save_figure(fig3, f"{model_name}_pr_curves", subdir)

    # ── Per-class F1 bar chart ────────────────────────────────────────────────
    fig4, ax4 = plt.subplots(figsize=(9, 5))
    bars = ax4.bar(
        TARGET_EMOTIONS,
        [metrics["f1_per_class"][e] for e in TARGET_EMOTIONS],
        color="mediumpurple", edgecolor="white",
    )
    ax4.bar_label(bars, fmt="%.3f", padding=3)
    ax4.set_ylim(0, 1.1)
    ax4.set_ylabel("F1 Score")
    ax4.set_title(f"Per-Class F1 — {model_name}")
    plt.tight_layout()
    artifact_manager.save_figure(fig4, f"{model_name}_f1_per_class", subdir)

    # ── Gate distribution (if available) ─────────────────────────────────────
    if gate_vals is not None:
        fig5, ax5 = plt.subplots(figsize=(8, 4))
        ax5.hist(gate_vals, bins=30, color="steelblue", edgecolor="white")
        ax5.axvline(
            gate_vals.mean(), color="red", linestyle="--",
            label=f"Mean={gate_vals.mean():.3f}",
        )
        ax5.set_xlabel("Gate Value (face weight)")
        ax5.set_ylabel("Count")
        ax5.set_title(f"Attention Gate Distribution — {model_name}")
        ax5.legend()
        plt.tight_layout()
        artifact_manager.save_figure(
            fig5, f"{model_name}_gate_distribution", subdir
        )

    _log.info(f"Joint evaluation complete → {subdir}")
    return metrics


# =============================================================================
# 14.  BEHAVIOURAL ENGINE
# =============================================================================

class BehavioralEngine:
    INTERVENTIONS = {
        "anger": {
            "message": "I can see you're feeling angry. Let's try counting to ten together.",
            "tasks":   ["Deep Breathing x 3", "Squeeze Stress Ball", "Calm Corner Break"],
            "cues":    ["Lower your voice", "Open hands", "Count slowly"],
        },
        "fear": {
            "message": "It's okay to feel scared — you are completely safe right now.",
            "tasks":   ["5-4-3-2-1 Grounding", "Weighted Blanket", "Check Safe Zone"],
            "cues":    ["Feet flat on floor", "Look around slowly", "Name 5 things you see"],
        },
        "joy": {
            "message": "I love that smile — amazing work today! Keep going!",
            "tasks":   ["Verbal Praise", "High Five", "Sticker Reward"],
            "cues":    ["Share the excitement", "Channel energy positively", "Celebrate!"],
        },
        "sadness": {
            "message": "It's okay to feel sad. Would you like some calming music?",
            "tasks":   ["Calming Music", "Comfort Object", "Quiet Sensory Space"],
            "cues":    ["Sit beside them", "Speak gently", "Offer favourite item"],
        },
        "surprise": {
            "message": "Wow, something unexpected! Let's take a breath together.",
            "tasks":   ["Wait Time (30 s)", "Reset Routine", "Preview Next Step"],
            "cues":    ["Reduce stimuli", "Use calm voice", "Show schedule"],
        },
        "Natural": {
            "message": "You're doing a wonderful job staying calm and focused!",
            "tasks":   ["Schedule Review", "Next Activity Preview", "Positive Reinforcement"],
            "cues":    ["Maintain routine", "Offer choice", "Praise calmness"],
        },
    }
    LOCATION_RULES = {
        "school":   "School environment — follow classroom protocol.",
        "home":     "Home environment — relaxed routine preferred.",
        "therapy":  "Therapy session — therapist-led strategies.",
        "outdoor":  "Outdoor — watch for sensory overload triggers.",
    }
    ROUTINE_RULES = {
        "morning routine": "Begin with visual schedule review.",
        "classroom":       "Use quiet signals; avoid disruption.",
        "lunch break":     "Allow unstructured wind-down time.",
        "therapy session": "Follow therapist protocol closely.",
    }
    TIME_RULES = {
        "morning":   "Energy building — set clear expectations.",
        "afternoon": "Possible fatigue — watch for frustration.",
        "evening":   "Wind-down mode — favour calming activities.",
    }

    def __init__(self, base_dir: Path):
        self._log        = logging.getLogger("EmotionAI.BehavioralEngine")
        self.base_dir    = base_dir
        self.memory_path = base_dir / "data/memory/interaction_history.json"
        self.policy_path = base_dir / "data/memory/reinforcement_policy.json"
        self.memory      = self._load_json(self.memory_path, [])
        self.policy      = self._load_json(
            self.policy_path,
            {
                "sensitivity":   1.0,
                "feedback_count": 0,
                "emotion_freq":  {e: 0 for e in TARGET_EMOTIONS},
            },
        )

    def _load_json(self, path: Path, default: Any) -> Any:
        if path.exists():
            try:
                with open(path) as f:
                    return json.load(f)
            except Exception as exc:
                self._log.warning(f"Load failed {path}: {exc}")
        return default

    def _save_json(self, path: Path, data: Any) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        try:
            with open(path, "w") as f:
                json.dump(data, f, indent=4)
        except IOError as exc:
            self._log.error(f"Save failed {path}: {exc}")

    def _update_sensitivity(self, rating: int) -> None:
        self.policy["sensitivity"] = float(
            np.clip(self.policy["sensitivity"] + (rating - 3) * 0.05, 0.5, 2.0)
        )
        self.policy["feedback_count"] += 1
        self._save_json(self.policy_path, self.policy)

    def save_interaction(self, data: dict) -> None:
        data["timestamp"] = str(datetime.datetime.now())
        em = data.get("emotion", "Natural")
        self.policy["emotion_freq"][em] = self.policy["emotion_freq"].get(em, 0) + 1
        self.memory.append(data)
        try:
            self._save_json(
                self.memory_path, {"history": self.memory[-200:]}
            )
            self._save_json(self.policy_path, self.policy)
            if "feedback" in data:
                self._update_sensitivity(int(data["feedback"]))
        except Exception as exc:
            self._log.error(f"Persist failed: {exc}", exc_info=True)

    def get_intervention(
        self,
        emotion: str,
        location: str = "",
        routine: str = "",
        time_of_day: str = "",
    ) -> dict:
        base    = self.INTERVENTIONS.get(emotion, self.INTERVENTIONS["Natural"])
        ctx     = []
        loc_key  = next((k for k in self.LOCATION_RULES if k in location.lower()),    None)
        rtn_key  = next((k for k in self.ROUTINE_RULES  if k in routine.lower()),     None)
        time_key = next((k for k in self.TIME_RULES     if k in time_of_day.lower()), None)
        if loc_key:  ctx.append(self.LOCATION_RULES[loc_key])
        if rtn_key:  ctx.append(self.ROUTINE_RULES[rtn_key])
        if time_key: ctx.append(self.TIME_RULES[time_key])
        hist = [r.get("emotion") for r in self.memory[-5:]]
        if hist.count(emotion) >= 3:
            ctx.append(f"Repeated '{emotion}' — consider escalating support.")
        return {
            "message":     base["message"],
            "tasks":       list(base["tasks"]),
            "cues":        list(base["cues"]),
            "context":     ctx,
            "sensitivity": self.policy["sensitivity"],
        }

    def speak(self, text: str) -> str:
        ts   = datetime.datetime.now().strftime("%H%M%S_%f")
        path = self.base_dir / f"outputs/audio/speech_{ts}.mp3"
        path.parent.mkdir(parents=True, exist_ok=True)
        try:
            gTTS(text=text, lang="en", slow=False).save(str(path))
            return str(path)
        except Exception as exc:
            self._log.error(f"TTS failed: {exc}")
            return ""

    def generate_schedule_card(self, tasks: List[str], emotion: str) -> str:
        COLORS = {
            "anger":    (200,  60,  60),
            "fear":     ( 80,  80, 200),
            "joy":      ( 60, 180,  60),
            "sadness":  ( 80, 140, 200),
            "surprise": (200, 140,  40),
            "Natural":  ( 80, 160, 120),
        }
        bg  = COLORS.get(emotion, (120, 120, 120))
        img = np.ones((500, 700, 3), dtype=np.uint8) * 245
        cv2.rectangle(img, (0, 0), (700, 70), bg, -1)
        cv2.putText(
            img, f"AI Support Plan — {emotion.upper()}",
            (20, 48), cv2.FONT_HERSHEY_DUPLEX, 1.0, (255, 255, 255), 2,
        )
        for i, t in enumerate(tasks):
            y = 120 + i * 75
            cv2.rectangle(img, (30, y - 30), (670, y + 30), (220, 220, 220), -1)
            cv2.rectangle(img, (30, y - 30), (670, y + 30), bg, 2)
            cv2.putText(
                img, f"{i+1}. {t}",
                (50, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (30, 30, 30), 2,
            )
        ts   = datetime.datetime.now().strftime("%H%M%S_%f")
        path = self.base_dir / f"outputs/schedules/schedule_{emotion}_{ts}.png"
        path.parent.mkdir(parents=True, exist_ok=True)
        try:
            ok = cv2.imwrite(str(path), img)
            if not ok:
                raise IOError("cv2.imwrite returned False")
            return str(path)
        except Exception as exc:
            self._log.error(f"Schedule card failed: {exc}")
            return ""


# =============================================================================
# 15.  TRAINING FUNCTIONS  (FIX-04 / FIX-08 / FIX-09)
# =============================================================================

def _mixup_batch(
    X: np.ndarray, y: np.ndarray, alpha: float = 0.2
) -> Tuple[np.ndarray, np.ndarray]:
    lam  = np.random.beta(alpha, alpha, size=(len(X), 1, 1, 1)).astype(np.float32)
    idx  = np.random.permutation(len(X))
    lam2 = lam.squeeze()[:, np.newaxis]
    return (
        (lam * X + (1 - lam) * X[idx]).astype(np.float32),
        (lam2 * y + (1 - lam2) * y[idx]).astype(np.float32),
    )


def train_vision_model(
    vision_model:  tf.keras.Model,
    train_gen,
    val_gen,
    class_weights: dict,
    base_dir:      Path,
) -> tf.keras.Model:
    _log = logging.getLogger("EmotionAI.training")
    _assert_keras_model(vision_model, "vision_model")

    _log.info("STAGE 1 — Vision warm-up (frozen base, 15 epochs)")
    vision_model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=15,
        class_weight=class_weights,
        callbacks=get_standard_callbacks(
            base_dir / "models/checkpoints/vision/stage1",
            use_reduce_lr=True,
        ),
    )

    _log.info("STAGE 2 — Vision fine-tune (top-60 EfficientNet layers, CosineDecay)")
    eff_base = vision_model.get_layer("efficientnetb0")
    eff_base.trainable = True
    total = len(eff_base.layers)
    for i, layer in enumerate(eff_base.layers):
        layer.trainable = (i >= total - 60)

    steps = max(1, train_gen.n // 32)
    cosine_decay = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=3e-5,
        decay_steps=steps * 10,
        alpha=1e-6,
    )
    vision_model.compile(
        optimizer=tf.keras.optimizers.Adam(cosine_decay),
        loss=tf.keras.losses.CategoricalCrossentropy(
            from_logits=True, label_smoothing=0.1
        ),
        metrics=["accuracy"],
    )

    vision_model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=10,
        class_weight=class_weights,
        callbacks=get_standard_callbacks(
            base_dir / "models/checkpoints/vision/stage2",
            use_reduce_lr=False,
        ),
    )

    return vision_model


def train_audio_model(
    audio_model: tf.keras.Model,
    X_train:     np.ndarray,
    y_train:     np.ndarray,
    X_val:       np.ndarray,
    y_val:       np.ndarray,
    base_dir:    Path,
) -> tf.keras.Model:
    _log = logging.getLogger("EmotionAI.training")
    _assert_keras_model(audio_model, "audio_model")
    _log.info("STAGE 3 — Audio training (50 epochs + MixUp)")

    y_tr_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
    y_v_cat  = tf.keras.utils.to_categorical(y_val,   NUM_CLASSES)
    cls_w    = compute_class_weight(
        "balanced", classes=np.unique(y_train), y=y_train
    )
    batch_sz    = 32
    audio_shape = AUDIO_SHAPE

    def make_dataset(X: np.ndarray, y: np.ndarray, augment: bool = False):
        ds = tf.data.Dataset.from_tensor_slices(
            (X.astype(np.float32), y.astype(np.float32))
        )
        if augment:
            def _apply_mixup(xb, yb):
                xm, ym = tf.py_function(
                    func=lambda a, b: _mixup_batch(a.numpy(), b.numpy()),
                    inp=[xb, yb],
                    Tout=[tf.float32, tf.float32],
                )
                xm.set_shape([None, *audio_shape])
                ym.set_shape([None, NUM_CLASSES])
                return xm, ym

            ds = (
                ds.shuffle(len(X), seed=SEED)
                  .batch(batch_sz, drop_remainder=True)
                  .map(_apply_mixup, num_parallel_calls=tf.data.AUTOTUNE)
                  .prefetch(tf.data.AUTOTUNE)
            )
        else:
            ds = ds.batch(batch_sz).prefetch(tf.data.AUTOTUNE)
        return ds

    audio_model.fit(
        make_dataset(X_train, y_tr_cat, augment=True),
        validation_data=make_dataset(X_val, y_v_cat),
        epochs=50,
        class_weight=dict(enumerate(cls_w)),
        callbacks=get_standard_callbacks(
            base_dir / "models/checkpoints/audio",
            use_reduce_lr=True,
        ),
    )

    return audio_model


def train_joint_model(
    joint_train_model: tf.keras.Model,
    train_gen,
    val_gen,
    X_audio_train:     np.ndarray,
    y_audio_train:     np.ndarray,
    X_audio_val:       np.ndarray,
    y_audio_val:       np.ndarray,
    base_dir:          Path,
) -> tf.keras.Model:
    _log = logging.getLogger("EmotionAI.training")
    _assert_keras_model(joint_train_model, "joint_train_model")
    _log.info("STAGE 4 — Joint multimodal training (10 epochs)")

    y_tr_cat = tf.keras.utils.to_categorical(
        y_audio_train, NUM_CLASSES
    ).astype(np.float32)
    y_v_cat = tf.keras.utils.to_categorical(
        y_audio_val, NUM_CLASSES
    ).astype(np.float32)

    def joint_generator(face_gen, X_audio: np.ndarray, y_cat: np.ndarray):
        face_gen.reset()
        face_iter = iter(face_gen)
        while True:
            try:
                fb, fl = next(face_iter)
            except StopIteration:
                face_gen.reset()
                face_iter = iter(face_gen)
                fb, fl   = next(face_iter)

            bsz = len(fb)
            idx = np.random.choice(len(X_audio), bsz, replace=True)
            jl = (
                0.5 * fl.astype(np.float32) + 0.5 * y_cat[idx]
            ).astype(np.float32)

            yield (
                {
                    "joint_face_input":  fb.astype(np.float32),
                    "joint_audio_input": X_audio[idx].astype(np.float32),
                },
                jl,
            )

    train_gen.reset()
    val_gen.reset()
    tg = joint_generator(train_gen, X_audio_train, y_tr_cat)
    vg = joint_generator(val_gen,   X_audio_val,   y_v_cat)

    _x, _y = next(tg)
    assert _y.dtype == np.float32, f"Expected float32 labels, got {_y.dtype}"
    _log.info("Joint generator pre-flight OK.")

    joint_train_model.fit(
        tg,
        steps_per_epoch=max(1, train_gen.n // 32),
        validation_data=vg,
        validation_steps=max(1, val_gen.n // 32),
        epochs=10,
        callbacks=get_standard_callbacks(
            base_dir / "models/checkpoints/joint",
            monitor="val_accuracy",
            use_reduce_lr=True,
        ),
    )

    return joint_train_model


# =============================================================================
# 16.  PRODUCTION ARTIFACT SAVING  (FIX-02 / FIX-05)
# =============================================================================

def save_production_artifacts(
    vision_model:      tf.keras.Model,
    audio_model:       tf.keras.Model,
    joint_infer_model: tf.keras.Model,
    base_dir:          Path,
) -> Path:
    _log      = logging.getLogger("EmotionAI.export")
    final_dir = base_dir / "models/final"
    final_dir.mkdir(parents=True, exist_ok=True)

    save_model_keras(
        vision_model, final_dir / "vision_model.keras", "VisionModel"
    )
    save_model_keras(
        audio_model,  final_dir / "audio_model.keras",  "AudioModel"
    )
    save_model_keras(
        joint_infer_model,
        final_dir / "joint_infer_model.keras",
        "JointInferModel",
    )

    meta = {
        "version":        "7.0",
        "emotions":       TARGET_EMOTIONS,
        "num_classes":    NUM_CLASSES,
        "img_size":       IMG_SIZE,
        "audio_shape":    list(AUDIO_SHAPE),
        "ravdess_to_asd": RAVDESS_TO_ASD,
        "saved_at":       str(datetime.datetime.now()),
    }
    with open(final_dir / "class_indices.json", "w") as f:
        json.dump({str(k): v for k, v in IDX_TO_EMOTION.items()}, f, indent=4)
    with open(final_dir / "metadata.json", "w") as f:
        json.dump(meta, f, indent=4)
    with open(final_dir / "MODEL_CARD.md", "w") as f:
        f.write(
            "---\nlicense: mit\ntags:\n  - emotion-recognition\n"
            "  - autism\n  - multimodal\n---\n\n"
            f"# Multimodal Emotion AI v7.0\n"
            f"**Emotions**: {', '.join(TARGET_EMOTIONS)}\n\n"
            "**Inputs**: Facial image (224×224 RGB) + 3-ch MFCC (40×128)\n\n"
            "**Explainability**: Grad-CAM (4 artefacts) + SHAP\n"
            "**Format**: Keras native (.keras)\n"
        )

    _log.info(f"All production artefacts saved → {final_dir}")
    return final_dir


# =============================================================================
# 17.  GRADIO UI
# =============================================================================

def launch_gradio_app(
    vision_model:      tf.keras.Model,
    audio_model:       tf.keras.Model,
    joint_infer_model: tf.keras.Model,
    grad_cam:          GradCAMExplainer,
    shap_explainer:    SHAPAudioExplainer,
    engine:            BehavioralEngine,
    base_dir:          Path,
    debug_mode:        bool = False,
) -> None:
    _log = logging.getLogger("EmotionAI.GradioUI")
    _log.info("Launching Gradio UI v7.0 …")

    vision_prob = build_softmax_wrapper(vision_model, "VisionModel_ui_prob")
    audio_prob  = build_softmax_wrapper(audio_model,  "AudioModel_ui_prob")

    def _empty(status):
        return (status, {}, "—", "—", None, None, "")

    def run_inference(
        image_np, audio_path, routine, time_of_day, location, feedback_rating
    ):
        if image_np is None:
            return _empty("⚠️ No image provided.")
        if image_np.ndim != 3 or image_np.shape[2] != 3:
            return _empty(f"⚠️ Invalid image shape {image_np.shape}.")

        try:
            raw_rgb_u8 = cv2.resize(image_np, (IMG_SIZE, IMG_SIZE))
            if raw_rgb_u8.dtype != np.uint8:
                raw_rgb_u8 = np.clip(raw_rgb_u8, 0, 255).astype(np.uint8)

            img_f32 = tf.keras.applications.efficientnet.preprocess_input(
                raw_rgb_u8.astype(np.float32)
            )
            img_batch = img_f32[np.newaxis]

            face_probs = vision_prob.predict(img_batch, verbose=0)[0]

            has_audio = False
            mfcc      = np.zeros(AUDIO_SHAPE, np.float32)
            xcam_msg  = ""

            if audio_path and Path(audio_path).exists():
                feat = extract_mfcc_spectrogram(audio_path)
                if feat is not None:
                    mfcc      = feat
                    has_audio = True
                else:
                    xcam_msg = "⚠️ MFCC extraction failed — SHAP skipped."
            else:
                xcam_msg = "ℹ️ No audio provided — SHAP skipped."

            audio_batch = mfcc[np.newaxis]
            audio_probs = audio_prob.predict(audio_batch, verbose=0)[0]

            joint_out = joint_infer_model.predict(
                {
                    "joint_face_input":  img_batch,
                    "joint_audio_input": audio_batch,
                },
                verbose=0,
            )
            if isinstance(joint_out, (list, tuple)) and len(joint_out) >= 2:
                gate_val = np.array(joint_out[1])
            else:
                _log.warning("Unexpected joint_infer output — using fallback gate 0.7")
                gate_val = np.array([[0.7]])
            learned_gate = float(gate_val.flatten()[0])

            img_bgr_q  = cv2.cvtColor(raw_rgb_u8, cv2.COLOR_RGB2BGR)
            img_quality = HybridFusion.assess_image_quality(img_bgr_q)
            fused, face_w, audio_w = HybridFusion.fuse(
                face_probs, audio_probs, learned_gate, img_quality
            )

            sens   = engine.policy.get("sensitivity", 1.0)
            fused  = np.power(fused, sens)
            fused /= fused.sum() + 1e-9

            pred_idx   = int(np.argmax(fused))
            emotion    = IDX_TO_EMOTION[pred_idx]
            confidence = float(fused[pred_idx])

            gcam_msg = ""
            try:
                gr_result = grad_cam.compute(
                    img_array=img_batch,
                    raw_rgb=raw_rgb_u8,
                    class_idx=pred_idx,
                )
                gcam_msg = (
                    f"✅ Grad-CAM saved (4 artefacts):\n"
                    f"  📊 {gr_result['report_path']}\n"
                    f"  🌡️ {gr_result['raw_heatmap_path']}\n"
                    f"  🎨 {gr_result['colour_heatmap_path']}\n"
                    f"  🔲 {gr_result['overlay_path']}"
                )
            except Exception as exc:
                gcam_msg = (
                    f"⚠️ Grad-CAM failed: {type(exc).__name__}: {exc}"
                )
                _log.error("Grad-CAM error: %s", exc, exc_info=True)

            shap_msg = xcam_msg
            if has_audio:
                try:
                    sr = shap_explainer.explain(mfcc, true_label=pred_idx)
                    shap_msg = f"✅ SHAP saved → {sr['plot_path']}"
                except Exception as exc:
                    shap_msg = f"⚠️ SHAP failed: {type(exc).__name__}."
                    _log.error("SHAP error: %s", exc, exc_info=True)

            interv     = engine.get_intervention(
                emotion, location, routine, time_of_day
            )
            tts_audio  = engine.speak(interv["message"]) or None
            sched_path = engine.generate_schedule_card(interv["tasks"], emotion)
            sched_img  = None
            if sched_path:
                sb = cv2.imread(sched_path)
                if sb is not None:
                    sched_img = cv2.cvtColor(sb, cv2.COLOR_BGR2RGB)

            engine.save_interaction({
                "emotion":       emotion,
                "confidence":    confidence,
                "face_weight":   round(face_w,       3),
                "audio_weight":  round(audio_w,      3),
                "image_quality": img_quality,
                "learned_gate":  round(learned_gate, 3),
                "location":      location,
                "routine":       routine,
                "time":          time_of_day,
                "feedback":      int(feedback_rating),
                "modality":      "Face+Audio" if has_audio else "Face+ZeroAudio",
            })

            status_msg = (
                f"### Detected: **{emotion.upper()}** ({confidence:.1%})\n\n"
                f"**Quality:** {img_quality}  |  "
                f"**Fusion:** Face {face_w:.0%} / Audio {audio_w:.0%} "
                f"(Gate {learned_gate:.2f})\n\n"
                f"**Context:** {location} | {time_of_day} | {routine}"
            )
            strategy = (
                f"**Message:** {interv['message']}\n\n"
                "**Tasks:**\n" + "\n".join(f"- {t}" for t in interv["tasks"])
                + "\n\n**Cues:**\n"
                + "\n".join(f"- {c}" for c in interv["cues"])
                + (
                    "\n\n**Context Notes:**\n"
                    + "\n".join(f"- {n}" for n in interv["context"])
                    if interv.get("context") else ""
                )
            )
            metrics_text = (
                f"**Metrics**\n"
                f"- Sensitivity: {sens:.2f}\n"
                f"- Feedback count: {engine.policy.get('feedback_count', 0)}\n"
                f"- Sessions: {len(engine.memory)}\n"
                f"- Quality: {img_quality} | Gate: {learned_gate:.4f}\n"
                f"- Freq: "
                + ", ".join(
                    f"{e}:{engine.policy['emotion_freq'].get(e, 0)}"
                    for e in TARGET_EMOTIONS
                )
            )
            xai_summary = f"{gcam_msg}\n\n{shap_msg}".strip()

            return (
                status_msg,
                {IDX_TO_EMOTION[i]: float(fused[i]) for i in range(NUM_CLASSES)},
                strategy,
                metrics_text,
                sched_img,
                tts_audio,
                xai_summary,
            )

        except Exception as exc:
            _log.error("Inference error:\n%s", traceback.format_exc())
            return (
                f"### ❌ Error\n```\n{exc}\n```",
                {}, "—", "—", None, None, "",
            )

    with gr.Blocks(
        theme=gr.themes.Soft(),
        title="Multimodal Emotion AI v7.0 — ASD Support",
    ) as demo:
        gr.Markdown(
            "# 🧠 Multimodal Emotion AI v7.0 — ASD Support\n"
            "*Real-time facial + vocal emotion recognition with adaptive "
            "behavioural support.*\n\n"
            "> **Explainability**: Grad-CAM (4 artefacts) + SHAP auto-saved "
            "to disk after every inference. Paths shown in the panel below."
        )
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📥 Input")
                img_in   = gr.Image(label="Facial Image", type="numpy")
                audio_in = gr.Audio(
                    label="Vocal Input",
                    type="filepath",
                    sources=["microphone", "upload"],
                )
                routine  = gr.Dropdown(
                    choices=[
                        "Morning Routine", "Classroom",
                        "Lunch Break", "Therapy Session",
                    ],
                    label="Current Routine",
                    value="Classroom",
                )
                time_day = gr.Radio(
                    choices=["Morning", "Afternoon", "Evening"],
                    label="Time of Day",
                    value="Morning",
                )
                location = gr.Textbox(
                    label="Location / Environment", value="School"
                )
                feedback = gr.Slider(
                    minimum=1, maximum=5, value=3, step=1,
                    label="Intervention Effectiveness (1=poor, 5=excellent)",
                )
                btn = gr.Button(
                    "🔍 Analyse Multimodal Signals", variant="primary"
                )

            with gr.Column(scale=2):
                gr.Markdown("### 📊 Results")
                status_out   = gr.Markdown("*Awaiting input…*")
                emotion_out  = gr.Label(label="Emotion Probabilities (Fused)")
                strategy_out = gr.Markdown(label="Adaptive Strategy")
                metrics_out  = gr.Markdown(label="System Metrics")

        with gr.Row():
            sched_out = gr.Image(label="🗓️ Visual Schedule Card", type="numpy")
            audio_out = gr.Audio(label="🔊 Audio Guidance (TTS)")

        with gr.Row():
            xai_out = gr.Markdown(
                label="🔬 Explainability Artefacts (saved to disk)", value=""
            )

        btn.click(
            fn=run_inference,
            inputs=[img_in, audio_in, routine, time_day, location, feedback],
            outputs=[
                status_out, emotion_out, strategy_out,
                metrics_out, sched_out, audio_out, xai_out,
            ],
        )

    demo.launch(share=True)


# =============================================================================
# 18.  MAIN ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    BASE_DIR = Path(f"/content/{PROJECT_NAME}")

    logger = setup_logging(BASE_DIR / "logs")
    logger.info("=" * 60)
    logger.info("  Multimodal Emotion AI — v7.0  (full rewrite, all fixes)")
    logger.info("=" * 60)

    generate_project_structure(BASE_DIR)
    artifacts = ArtifactManager(BASE_DIR)

    AUDIO_ZIP  = "/content/Audio_Speech_Actors_01-24_16k.zip"
    FACIAL_ZIP = "/content/Autistic Children Emotions - Dr. Fatma M. Talaat 2.zip"

    # ── Data ──────────────────────────────────────────────────────────────────
    X_audio, y_audio = load_audio_dataset(AUDIO_ZIP, BASE_DIR)

    perm  = np.random.permutation(len(X_audio))
    split = int(0.8 * len(X_audio))
    X_audio_train = X_audio[perm[:split]]
    y_audio_train = y_audio[perm[:split]]
    X_audio_val   = X_audio[perm[split:]]
    y_audio_val   = y_audio[perm[split:]]

    train_gen, val_gen, face_cls_w = load_facial_dataset(FACIAL_ZIP, BASE_DIR)

    # ── Build models ──────────────────────────────────────────────────────────
    vision_model = build_vision_model()
    audio_model  = build_audio_model()
    joint_train_model, joint_infer_model = build_joint_model(
        vision_model, audio_model
    )

    for m, lbl in [
        (vision_model,       "vision"),
        (audio_model,        "audio"),
        (joint_train_model,  "joint_train"),
        (joint_infer_model,  "joint_infer"),
    ]:
        _assert_keras_model(m, lbl)
    logger.info("All model instances validated.")

    # ── Train ─────────────────────────────────────────────────────────────────
    vision_model      = train_vision_model(
        vision_model, train_gen, val_gen, face_cls_w, BASE_DIR
    )
    audio_model       = train_audio_model(
        audio_model,
        X_audio_train, y_audio_train,
        X_audio_val,   y_audio_val,
        BASE_DIR,
    )
    joint_train_model = train_joint_model(
        joint_train_model,
        train_gen, val_gen,
        X_audio_train, y_audio_train,
        X_audio_val,   y_audio_val,
        BASE_DIR,
    )

    # ── Evaluation ────────────────────────────────────────────────────────────
    X_face_val, y_face_val = collect_val_arrays(val_gen, n_total=val_gen.n)
    evaluate_and_report(
        vision_model, X_face_val, y_face_val, artifacts, "VisionModel"
    )
    evaluate_and_report(
        audio_model, X_audio_val, y_audio_val, artifacts, "AudioModel"
    )

    # ── Joint Evaluation (NEW) ────────────────────────────────────────────────
    evaluate_joint_model(
        joint_infer_model=joint_infer_model,
        X_face_val=X_face_val,
        X_audio_val=X_audio_val,
        y_true=y_audio_val,
        artifact_manager=artifacts,
        model_name="JointModel",
    )

    # ── Explainability ────────────────────────────────────────────────────────
    logger.info("Initialising GradCAMExplainer …")
    grad_cam = GradCAMExplainer(vision_model, artifacts)

    logger.info("Initialising SHAPAudioExplainer …")
    shap_exp = SHAPAudioExplainer(audio_model, X_audio_train, artifacts)
    shap_exp.batch_shap_report(X_audio_val, y_audio_val, n_samples=30)

    demo_X, demo_y = collect_val_arrays(val_gen, n_total=32)
    if len(demo_X) > 0:
        sp = demo_X[0]
        sr = np.clip(((sp / 2.0) + 64.0), 0, 255).astype(np.uint8)
        try:
            res = grad_cam.compute(
                img_array=sp[np.newaxis],
                raw_rgb=sr,
                class_idx=int(demo_y[0]),
            )
            logger.info(
                f"Demo Grad-CAM: {res['emotion']} ({res['confidence']:.2%})"
            )
        except Exception as exc:
            logger.warning(f"Demo Grad-CAM skipped: {exc}")

    # ── Save all models ───────────────────────────────────────────────────────
    save_production_artifacts(
        vision_model, audio_model, joint_infer_model, BASE_DIR
    )

    # ── Launch UI ─────────────────────────────────────────────────────────────
    engine = BehavioralEngine(BASE_DIR)
    launch_gradio_app(
        vision_model,
        audio_model,
        joint_infer_model,
        grad_cam,
        shap_exp,
        engine,
        BASE_DIR,
        debug_mode=False,
    )

2026-04-04 20:17:56 | INFO     | EmotionAI                 | Logging initialised → /content/EmotionallyAdaptiveAI/logs/pipeline.log


INFO:EmotionAI:Logging initialised → /content/EmotionallyAdaptiveAI/logs/pipeline.log


2026-04-04 20:17:56 | INFO     | EmotionAI                 | ============================================================


INFO:EmotionAI:============================================================


2026-04-04 20:17:56 | INFO     | EmotionAI                 |   Multimodal Emotion AI — v7.0  (full rewrite, all fixes)


...

2026-04-04 20:29:48 | INFO     | EmotionAI.export          | All production artefacts saved → /content/EmotionallyAdaptiveAI/models/final


INFO:EmotionAI.export:All production artefacts saved → /content/EmotionallyAdaptiveAI/models/final


2026-04-04 20:29:48 | INFO     | EmotionAI.GradioUI        | Launching Gradio UI v7.0 …


INFO:EmotionAI.GradioUI:Launching Gradio UI v7.0 …


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0cee8d960bc6013298.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
